# Public-data quality and metric checks

## tl;dr

The executed input contains 2,678 trajectories, 689 subjects and 320,080 trials. No duplicate records/trajectory IDs, missing IDs/rewards, invalid actions or non-finite rewards were found. Category denominators and the switch-loss decomposition reconcile with the frozen score. No raw trajectories or participant identifiers are printed.

## Context & Methods

Prepared 2026-08-27 for the final modeling package. Grain: one trajectory containing sequential trials; subjects are the grouping unit for validation. Input: `data/public_train.jsonl`. Reference: `artifacts/baseline_public_evaluation.json`.

### Key Assumptions

The supplied row order is the observed trial order. These checks cannot establish hidden-task generalization or independently verify the dataset's original collection process. The public data has already been used for model development.


## Data

Read only the public trajectories; compute a content hash and aggregate completeness, duplicate, action and reward checks. Standard-library Python is sufficient for the calculations.


In [1]:
import hashlib, json, math, random
from collections import Counter
from pathlib import Path
root = Path.cwd()
if not (root / 'config.yaml').exists():
    root = root.parent
data_path = root / 'data/public_train.jsonl'
records = [json.loads(line) for line in data_path.read_text(encoding='utf-8').splitlines() if line.strip()]
subjects = sorted({r['context']['subject_id'] for r in records})
keys = [r['context'].get('trajectory_id') for r in records]
canonical = [json.dumps(r, sort_keys=True, separators=(',', ':')) for r in records]
counts = Counter(first=0, stay=0, switch=0)
invalid_actions = missing_rewards = nonfinite_rewards = empty_trajectories = 0
action_counts = Counter()
for trajectory in records:
    actions = trajectory['context']['available_actions']
    assert actions and len(actions) == len(set(actions))
    action_counts[len(actions)] += 1
    trials = trajectory['trials']
    empty_trajectories += not bool(trials)
    for index, trial in enumerate(trials):
        invalid_actions += trial['action'] not in actions
        category = 'first' if index == 0 else ('stay' if trial['action'] == trials[index-1]['action'] else 'switch')
        counts[category] += 1
        reward = trial.get('reward')
        missing_rewards += reward is None
        if reward is not None:
            nonfinite_rewards += not math.isfinite(float(reward))
profile = dict(trajectories=len(records), subjects=len(subjects), trials=sum(counts.values()), categories=dict(counts), action_count_distribution=dict(action_counts), duplicate_full_records=len(records)-len(set(canonical)), missing_trajectory_ids=sum(k is None for k in keys), duplicate_nonmissing_trajectory_ids=len([k for k in keys if k is not None])-len({k for k in keys if k is not None}), invalid_actions=invalid_actions, missing_rewards=missing_rewards, nonfinite_rewards=nonfinite_rewards, empty_trajectories=empty_trajectories, data_sha256=hashlib.sha256(data_path.read_bytes()).hexdigest())
print(json.dumps(profile, indent=2))


{
  "trajectories": 2678,
  "subjects": 689,
  "trials": 320080,
  "categories": {
    "first": 2678,
    "stay": 239937,
    "switch": 77465
  },
  "action_count_distribution": {
    "4": 2678
  },
  "duplicate_full_records": 0,
  "missing_trajectory_ids": 0,
  "duplicate_nonmissing_trajectory_ids": 0,
  "invalid_actions": 0,
  "missing_rewards": 0,
  "nonfinite_rewards": 0,
  "empty_trajectories": 0,
  "data_sha256": "2a37588cfcbbc7e10df6b1eff22a23ed29a15bef490c18ed36811e27090f4230"
}


## Results

Check the saved score, the additive switch-loss decomposition and five-fold subject partition. The same subject must not appear on both sides of a fold. These are development checks, not a fresh test sample.


In [2]:
baseline = json.loads((root / 'artifacts/baseline_public_evaluation.json').read_text(encoding='utf-8'))
assert profile['data_sha256'] == baseline['data_sha256']
assert profile['trials'] == baseline['trials'] == 320080
assert profile['duplicate_full_records'] == profile['invalid_actions'] == profile['empty_trajectories'] == 0
assert profile['duplicate_nonmissing_trajectory_ids'] == 0
assert profile['missing_trajectory_ids'] == 0
for category, count in counts.items():
    assert baseline['categories'][category]['trials'] == count
weighted_nll = sum(v['trials'] * v['nll'] for v in baseline['categories'].values()) / baseline['trials']
assert abs(weighted_nll - baseline['nll']) < 1e-12
assert abs(baseline['switch_gate_nll'] + baseline['switch_target_nll'] - baseline['categories']['switch']['nll']) < 1e-12
shuffled = list(subjects)
random.Random(20260827).shuffle(shuffled)
folds = [set(shuffled[i::5]) for i in range(5)]
assert set.union(*folds) == set(subjects)
assert all(not folds[i] & folds[j] for i in range(5) for j in range(i))
print('All denominator, loss decomposition and grouping checks passed.')
print('Subjects per validation fold:', [len(fold) for fold in folds])
(root / 'artifacts/public_data_quality.json').write_text(json.dumps(profile, indent=2) + '\n', encoding='utf-8')


All denominator, loss decomposition and grouping checks passed.
Subjects per validation fold: [138, 138, 138, 138, 137]


492

## Takeaways

All executed checks passed. The first/stay/switch counts are 2,678 / 239,937 / 77,465; validation folds contain 138 / 138 / 138 / 138 / 137 subjects. No records are excluded or repaired. Trial-weighted scores are dominated by repeated choices. Prior exploration and model selection still limit generalization claims; no temporal freshness claim is made for this fixed benchmark extract.
